In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error

In [2]:
# 1. Setup and Data Partitioning
def prepare_federated_data(num_clients=5):
    data = fetch_california_housing()
    X, y = data.data, data.target
    
    # Preprocessing
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    # Split data among clients (non-iid simulation could be done here)
    X_shards = np.array_split(X, num_clients)
    y_shards = np.array_split(y, num_clients)
    
    return X_shards, y_shards

In [3]:
# 2. Client-Side Training
class Client:
    def __init__(self, client_id, X, y):
        self.client_id = client_id
        self.X = X
        self.y = y
        # Local model: Linear Regression via Stochastic Gradient Descent
        self.model = SGDRegressor(max_iter=10, tol=1e-3, learning_rate='constant', eta0=0.01)

    def local_train(self, global_weights, global_intercept):
        # Set local model to current global parameters
        if global_weights is not None:
            self.model.coef_ = global_weights
            self.model.intercept_ = global_intercept
        
        # Partial fit (one epoch of local training)
        self.model.partial_fit(self.X, self.y)
        
        return self.model.coef_, self.model.intercept_, len(self.X)

In [4]:
# 3. Server-Side Aggregation (FedAvg)
class Server:
    def __init__(self):
        self.global_weights = None
        self.global_intercept = None

    def aggregate(self, local_results):
        total_samples = sum(res[2] for res in local_results)
        
        # Weighted average of coefficients
        new_weights = np.zeros_like(local_results[0][0])
        new_intercept = np.zeros_like(local_results[0][1])
        
        for coef, intercept, num_samples in local_results:
            weight_factor = num_samples / total_samples
            new_weights += coef * weight_factor
            new_intercept += intercept * weight_factor
            
        self.global_weights = new_weights
        self.global_intercept = new_intercept



In [5]:
# --- Execution ---
num_clients = 5
rounds = 20
X_shards, y_shards = prepare_federated_data(num_clients)

# Initialize clients and server
clients = [Client(i, X_shards[i], y_shards[i]) for i in range(num_clients)]
server = Server()

print(f"Starting Federated Learning with {num_clients} clients...")

for r in range(rounds):
    local_updates = []
    
    for client in clients:
        # Each client trains locally starting from global parameters
        update = client.local_train(server.global_weights, server.global_intercept)
        local_updates.append(update)
    
    # Server aggregates updates to update global model
    server.aggregate(local_updates)
    
    # Evaluation (Optional: Evaluate global model on a test set)
    if r % 5 == 0 or r == rounds - 1:
        # Using first client's data as a dummy proxy for validation
        preds = np.dot(X_shards[0], server.global_weights) + server.global_intercept
        mse = mean_squared_error(y_shards[0], preds)
        print(f"Round {r}: Global Model MSE on Client 0 data: {mse:.4f}")

print("\nFinal Global Weights:", server.global_weights)

Starting Federated Learning with 5 clients...
Round 0: Global Model MSE on Client 0 data: 2831.8544
Round 5: Global Model MSE on Client 0 data: 480537518784742024019968.0000
Round 10: Global Model MSE on Client 0 data: 9064200679013586305024.0000
Round 15: Global Model MSE on Client 0 data: 140023873840142220263424.0000
Round 19: Global Model MSE on Client 0 data: 398185767341938675023872.0000

Final Global Weights: [-5.03847466e+08 -3.69470432e+09  2.15346124e+10 -2.01736930e+10
  7.70230071e+09 -7.03359306e+11  8.94345705e+09  9.74898895e+09]
